# 🧬 MiroFish-BioReviewer
### AI-Assisted Review of Systems Biology Grant Pre-Proposals

This notebook runs MiroFish-BioReviewer in Google Colab. It will:
1. Prompt you for API keys (stored only in session memory)
2. Accept your grant pre-proposal PDF upload
3. Build a knowledge graph from the proposal
4. Run a swarm simulation of biological entities from the proposal
5. Convene a 3-agent expert reviewer panel
6. Generate a structured review report

**Required API keys:** LLM provider key + ZEP Cloud key  
**Estimated runtime:** 5–15 minutes depending on model and simulation rounds  
**Default simulation length:** 40 rounds

> ⚠️ **Colab-only.** This notebook will refuse to run on local Jupyter, VS Code, or any non-Colab environment. If Colab's default runtime is Python 3.12 (or newer), Cell 2 automatically provisions a Python 3.11 venv at `/opt/mirofish_venv` because `camel-oasis` only supports Python 3.10–3.11. You don't need to do anything — just run the cells in order.

In [ ]:
# ── Cell 2: Environment Setup (Colab-only, auto Python 3.11 venv) ──────────
import os, sys, platform, subprocess

# 1. Sanity check: must be running in real Google Colab on Linux.
def _running_in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

_is_colab = _running_in_colab()
_is_linux = platform.system() == "Linux"
if not (_is_colab and _is_linux):
    raise RuntimeError(
        "❌ This notebook is designed to run in Google Colab on Linux only.\n"
        f"   Detected: platform={platform.system()}, in_colab={_is_colab}.\n\n"
        "Open the notebook via:\n"
        "  https://colab.research.google.com/github/kouroshSA/MiroFish-BioReviewer/blob/main/colab/MiroFish_BioReviewer.ipynb\n\n"
        "Local Windows/macOS execution is NOT supported because some required\n"
        "dependencies (camel-ai, camel-oasis) only ship pre-built wheels for Linux.\n"
        "That's the source of the 'Microsoft Visual C++ 14.0 or greater is required'\n"
        "error you may have just hit — pip is trying to compile from source on Windows."
    )

PY_MAJ, PY_MIN = sys.version_info[:2]
print(f"✅ Running in Colab on Linux (system Python {PY_MAJ}.{PY_MIN})")

# 2. Clone the repository if needed
if not os.path.exists("MiroFish-BioReviewer"):
    print("\n📥 Cloning MiroFish-BioReviewer...")
    subprocess.run([
        "git", "clone", "--depth=1",
        "https://github.com/kouroshSA/MiroFish-BioReviewer.git"
    ], check=True)

os.chdir("MiroFish-BioReviewer")
sys.path.insert(0, os.path.abspath("backend"))

# 3. Install system build dependencies via apt — required as a fallback path
#    in case any Python wheel needs to be compiled from source.
print("\n📦 Installing system build dependencies (apt)...")
subprocess.run(
    ["apt-get", "install", "-yqq",
     "build-essential", "poppler-utils", "libffi-dev"],
    check=False,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)

# 4. camel-oasis only ships wheels for Python 3.10–3.11 (every released
#    version pins requires_python="<3.12,>=3.10"). Colab now defaults to
#    Python 3.12, so we provision Python 3.11 via apt and run the pipeline
#    inside a venv. This is automatic — students do nothing.
NEED_VENV = (PY_MAJ, PY_MIN) >= (3, 12) or (PY_MAJ, PY_MIN) < (3, 10)
VENV_DIR = "/opt/mirofish_venv"

if NEED_VENV:
    print(
        f"\n⚠️  Colab runtime is Python {PY_MAJ}.{PY_MIN}, but camel-oasis "
        "requires Python 3.10 or 3.11.\n"
        f"📦 Installing Python 3.11 and creating a venv at {VENV_DIR} ..."
    )
    subprocess.run(
        ["apt-get", "install", "-yqq",
         "python3.11", "python3.11-venv", "python3.11-dev"],
        check=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.STDOUT,
    )
    if not os.path.exists(os.path.join(VENV_DIR, "bin", "python")):
        subprocess.run(["python3.11", "-m", "venv", VENV_DIR], check=True)
    target_py = os.path.join(VENV_DIR, "bin", "python")
else:
    target_py = sys.executable

print(f"📦 Pipeline interpreter: {target_py}")

# 5. Upgrade pip / wheel / setuptools inside the target interpreter so
#    prebuilt wheels are preferred.
print("\n📦 Upgrading pip / wheel / setuptools...")
subprocess.run(
    [target_py, "-m", "pip", "install", "-q", "--upgrade",
     "pip", "wheel", "setuptools"],
    check=True,
)

# 6. Install MiroFish-BioReviewer dependencies. We do NOT pass -q here so
#    that any failure surfaces immediately in the cell output instead of
#    being hidden behind a generic CalledProcessError.
print("\n📦 Installing MiroFish-BioReviewer Python dependencies (2-4 minutes)...\n")
ret = subprocess.run(
    [target_py, "-m", "pip", "install",
     "--prefer-binary",
     "-r", "backend/requirements.txt"],
)
if ret.returncode != 0:
    raise RuntimeError(
        "❌ pip install failed. The actual error is in the output above this "
        "traceback — scroll up to find which package and which line.\n\n"
        "Common causes:\n"
        "  • A package has no wheel for this Python version\n"
        "  • A transitive dependency conflicts with a pre-installed Colab pkg\n\n"
        "Try: Runtime → Restart runtime, then re-run this cell."
    )

# 7. Smoke-import critical packages via the target Python so any silent
#    install failure surfaces here, not in Cell 6.
print("\n🔎 Verifying critical packages import cleanly...")
_check_script = (
    "import sys\n"
    "pkgs = [('openai','openai'),\n"
    "        ('zep-cloud','zep_cloud'),\n"
    "        ('PyMuPDF','fitz'),\n"
    "        ('flask','flask'),\n"
    "        ('python-dotenv','dotenv'),\n"
    "        ('pydantic','pydantic'),\n"
    "        ('camel-ai','camel'),\n"
    "        ('camel-oasis','oasis')]\n"
    "missing = []\n"
    "for pretty, mod in pkgs:\n"
    "    try:\n"
    "        __import__(mod); print(f'   ✓ {pretty}')\n"
    "    except Exception as e:\n"
    "        print(f'   ✗ {pretty}: {type(e).__name__}: {e}'); missing.append(pretty)\n"
    "sys.exit(1 if missing else 0)\n"
)
ret = subprocess.run([target_py, "-c", _check_script])
if ret.returncode != 0:
    raise RuntimeError(
        "Critical packages failed to import (see ✗ rows above). "
        "Try Runtime → Restart runtime, then re-run this cell."
    )

# 8. Hand the venv interpreter to Cell 6 via env var.
os.environ["MIROFISH_PYTHON"] = target_py

print("\n✅ Setup complete.")
print(f"   Pipeline will run with: {target_py}")

In [ ]:
# ── Cell 3: API Key Configuration ──────────────────────────────────────────
# Keys are entered interactively and stored ONLY in session memory.
# They are never written to disk, never logged, and lost when the session ends.

import getpass
import os

print("🔑 API Key Configuration")
print("=" * 50)
print("Keys are stored in session memory only (getpass).")
print("They will NOT be saved to disk or appear in notebook output.\n")

# LLM Provider selection
print("Choose your LLM provider:")
print("  1. OpenAI (gpt-4o, gpt-4o-mini, gpt-4.1)")
print("  2. Anthropic Claude (via LiteLLM proxy — see README)")
print("  3. Google Gemini (gemini-2.5-flash)")
print("  4. DeepSeek (deepseek-chat)")
print("  5. Custom OpenAI-compatible endpoint")

provider_choice = input("\nEnter choice (1-5): ").strip()

PROVIDER_DEFAULTS = {
    "1": ("https://api.openai.com/v1", "gpt-4o-mini"),
    "2": ("http://localhost:8000/v1", "claude-sonnet-4-20250514"),
    "3": ("https://generativelanguage.googleapis.com/v1beta/openai", "gemini-2.5-flash"),
    "4": ("https://api.deepseek.com/v1", "deepseek-chat"),
    "5": (None, None),
}

default_base_url, default_model = PROVIDER_DEFAULTS.get(provider_choice, (None, None))

if provider_choice == "5":
    llm_base_url = input("Enter your custom base URL (e.g. http://localhost:11434/v1): ").strip()
    llm_model = input("Enter your model name: ").strip()
else:
    llm_base_url = default_base_url
    llm_model = input(f"Model name [{default_model}]: ").strip() or default_model

llm_api_key = getpass.getpass("\nEnter your LLM API key (hidden): ")

print("\nNow enter your ZEP Cloud API key.")
print("Get one free at: https://app.getzep.com/")
zep_api_key = getpass.getpass("ZEP API key (hidden): ")

# Set environment variables for the backend
os.environ["LLM_API_KEY"] = llm_api_key
os.environ["LLM_BASE_URL"] = llm_base_url
os.environ["LLM_MODEL_NAME"] = llm_model
os.environ["ZEP_API_KEY"] = zep_api_key
os.environ["SIMULATION_MODE"] = "grant_review"
os.environ["REVIEWER_PANEL_ENABLED"] = "true"

# Optional: stronger model for Discussion/Big Picture sections
use_strong_model = input(
    "\nUse a stronger model for final report synthesis? "
    "Enter model name or press Enter to use same model: "
).strip()
if use_strong_model:
    os.environ["DISCUSSION_LLM_MODEL_NAME"] = use_strong_model

print("\n✅ Configuration complete. Keys stored in session memory only.")
print(f"   Provider: {llm_base_url}")
print(f"   Model: {llm_model}")
if zep_api_key:
    print(f"   ZEP key: {'*' * (len(zep_api_key) - 4) + zep_api_key[-4:]}")

In [ ]:
# ── Cell 4: Upload Grant Pre-Proposal ──────────────────────────────────────
from google.colab import files
import os

print("📄 Upload your grant pre-proposal PDF (2–3 pages).")
print("Supported formats: PDF, Markdown (.md), Plain text (.txt)\n")

uploaded = files.upload()

if not uploaded:
    raise ValueError("No file uploaded. Please re-run this cell and upload a file.")

uploaded_filename = list(uploaded.keys())[0]
uploaded_path = os.path.abspath(uploaded_filename)
print(f"\n✅ Uploaded: {uploaded_filename} ({len(uploaded[uploaded_filename]):,} bytes)")

# Store path for downstream cells
os.environ["PROPOSAL_PATH"] = uploaded_path

In [ ]:
# ── Cell 5: Simulation Parameters ──────────────────────────────────────────
# Default rounds = 40 (Colab-specific default for MiroFish-BioReviewer)

max_rounds_input = input(
    "Max simulation rounds [default: 40, recommended: 20–60]: "
).strip()
max_rounds = int(max_rounds_input) if max_rounds_input.isdigit() else 40

simulation_request = input(
    "\nDescribe your review focus (or press Enter for default):\n"
    "Default: 'Review this systems biology grant pre-proposal. Evaluate scientific "
    "significance, innovation, mechanistic rigor, quantitative design, and feasibility.'\n> "
).strip()

if not simulation_request:
    simulation_request = (
        "Review this systems biology grant pre-proposal. "
        "Simulate how the key biological tools, systems, and targets described in "
        "the proposal would react to and debate the proposed research. "
        "Evaluate scientific significance, innovation, mechanistic rigor, "
        "quantitative and statistical design, and feasibility. "
        "Focus on entities described in the proposal — do NOT create agents for "
        "researchers, institutions, or journals."
    )

os.environ["MAX_ROUNDS"] = str(max_rounds)
os.environ["SIMULATION_REQUEST"] = simulation_request

print(f"\n✅ Parameters set: {max_rounds} rounds (default 40 in Colab)")

In [ ]:
# ── Cell 6: Run MiroFish-BioReviewer Pipeline ──────────────────────────────
# Runs the full pipeline using the Python interpreter Cell 2 selected
# (system Python on 3.10/3.11, or /opt/mirofish_venv/bin/python on 3.12+).

import subprocess, sys, os

output_dir = "colab_output"
os.makedirs(output_dir, exist_ok=True)

pipeline_python = os.environ.get("MIROFISH_PYTHON", sys.executable)

cmd = [
    pipeline_python, "backend/scripts/run_review_pipeline.py",
    "--proposal",   os.environ["PROPOSAL_PATH"],
    "--request",    os.environ["SIMULATION_REQUEST"],
    "--max-rounds", os.environ["MAX_ROUNDS"],
    "--output-dir", output_dir,
    "--mode",       "grant_review",
]
print("$ " + " ".join(cmd))
result = subprocess.run(cmd)

if result.returncode != 0:
    raise RuntimeError(f"Pipeline exited with status {result.returncode}")

os.environ["OUTPUT_DIR"] = output_dir
print("\n✅ Pipeline complete.")

In [ ]:
# ── Cell 7: Display and Download Report ────────────────────────────────────
import os, json
from IPython.display import Markdown, display
from google.colab import files

output_dir = os.environ.get("OUTPUT_DIR", "colab_output")

# Find the report file
report_candidates = ["full_report.md", "report.md", "LLM_Refined_Report.md"]
report_path = None
for candidate in report_candidates:
    p = os.path.join(output_dir, candidate)
    if os.path.exists(p):
        report_path = p
        break

if report_path:
    with open(report_path, "r", encoding="utf-8") as f:
        report_text = f.read()
    print(f"📋 Report found: {report_path}\n")
    display(Markdown(report_text))

    # Download
    download_confirm = input("\nDownload report as Markdown? (y/n): ").strip().lower()
    if download_confirm == "y":
        files.download(report_path)
else:
    print("⚠️  Report file not found. Check output directory:")
    print(os.listdir(output_dir))

# Also display reviewer panel if available
panel_path = os.path.join(output_dir, "reviewer_panel.json")
if os.path.exists(panel_path):
    with open(panel_path, "r", encoding="utf-8") as f:
        panel = json.load(f)

    print("\n\n📊 Reviewer Panel Raw Data:")
    for review in panel.get("reviews", []):
        print(f"\n{'='*50}")
        print(f"Reviewer: {review.get('reviewer', 'Unknown')}")
        print(f"Recommendation: {review.get('recommendation', 'N/A')}")
        scores = review.get("dimension_scores", {})
        for dim, score in scores.items():
            print(f"  {dim}: {score}/10")